In [ ]:
# This Python 3 environment comes with many helpful analytics libraries installed
# It is defined by the kaggle/python Docker image: https://github.com/kaggle/docker-python
# For example, here's several helpful packages to load

import numpy as np # linear algebra
import pandas as pd # data processing, CSV file I/O (e.g. pd.read_csv)

# Input data files are available in the read-only "../input/" directory
# For example, running this (by clicking run or pressing Shift+Enter) will list all files under the input directory

import os
for dirname, _, filenames in os.walk('/kaggle/input'):
    for filename in filenames:
        print(os.path.join(dirname, filename))

# You can write up to 20GB to the current directory (/kaggle/working/) that gets preserved as output when you create a version using "Save & Run All" 
# You can also write temporary files to /kaggle/temp/, but they won't be saved outside of the current session

In [ ]:
df=pd.read_csv('/kaggle/input/System-Threat-Forecaster/train.csv')

In [ ]:
df=pd.DataFrame(df)
df.dropna()
X = df.drop(columns=['target'])  # Features
y = df['target']  # Target variable

# **EDA**


In [ ]:
df.info()

**Histogram of numerical features**

In [ ]:
import seaborn as sns
import matplotlib.pyplot as plt
import numpy as np

num_cols = df.select_dtypes(include=['float64', 'int64']).columns
num_features = len(num_cols)
num_rows = int(np.ceil(num_features / 3))  

plt.figure(figsize=(12, num_rows * 3)) 

for i, col in enumerate(num_cols):
    plt.subplot(num_rows, 3, i + 1) 
    sns.histplot(df[col], kde=True, bins=30)
    plt.title(f'{col}')

plt.tight_layout()
plt.show()


**Correlation Heatmap for numerical features**

In [ ]:
import seaborn as sns
import matplotlib.pyplot as plt

# Correlation heatmap
numerical_df = df.select_dtypes(include=['float64','int64'])

corr_matrix = numerical_df.corr()

# Plot the correlation heatmap
plt.figure(figsize=(12, 12))
sns.heatmap(corr_matrix, annot=True, cmap='coolwarm', fmt=".2f")
plt.title('Correlation Heatmap of Numerical Features')
plt.show()

**Boxplot for outlier detection**

In [ ]:
num_rows=int(np.ceil(len(num_cols)/3))
plt.figure(figsize=(12, num_rows * 3)) 
for i, col in enumerate(num_cols):
    plt.subplot(num_rows, 3, i + 1)
    sns.boxplot(x=df[col])
    plt.title(f'Boxplot of {col}')
plt.tight_layout()
plt.show()

**Categorical Columns**

In [ ]:
import pandas as pd
import seaborn as sns
import matplotlib.pyplot as plt

cat_cols = [col for col in df.select_dtypes(include=['object']).columns if df[col].nunique()<20]
    
for col in cat_cols:
    plt.figure(figsize=(5, 3))
    sns.countplot(y=df[col], order=df[col].value_counts().index)
    plt.title(f"Distribution of {col}")
    plt.xlabel("Count")
    plt.ylabel(col)
    plt.show()

# Dummy Model 

In [ ]:
import numpy as np
from sklearn.model_selection import train_test_split
from sklearn.dummy import DummyClassifier
from sklearn.metrics import accuracy_score, classification_report,confusion_matrix
import matplotlib.pyplot as plt
import seaborn as sns

X = df.drop(columns=['target'])
y = df['target']
X_train,X_test,y_train,y_test=train_test_split(X,y,test_size=0.1,random_state=42)
# Initialize the DummyClassifier
dummy_clf = DummyClassifier(strategy='most_frequent').fit(X_train,y_train)  

# Make predictions
y_pred = dummy_clf.predict(X_test)
print(accuracy_score(y_test,y_pred))
print("Classification Report:\n", classification_report(y_test, y_pred))

conf_matrix = confusion_matrix(y_test, y_pred)

# Plot Confusion Matrix
plt.figure(figsize=(5, 4))
sns.heatmap(conf_matrix, annot=True, fmt='d', cmap='Blues', xticklabels=np.unique(y), yticklabels=np.unique(y))
plt.xlabel("Predicted Label")
plt.ylabel("True Label")
plt.title("Confusion Matrix")
plt.show()

# **Milestone 1**

In [ ]:
df1=df.dropna()
#1 How many unique versions of the operating system are present in the dataset?
print(df1["OSVersion"].unique().shape[0])

#2 What is the maximum value of the feature “NumAntivirusProductsInstalled”?
print(df1["NumAntivirusProductsInstalled"].max())

#3 In how many systems owned by gamers was malware detected? 
print(df1[(df1["IsGamer"]==1) & (df1["target"]==1)].shape[0])

#4For observations where the feature “IsPassiveModeEnabled” equals “1,” what is the most frequent value of the feature “RealTimeProtectionState”? 
print(df1[df1["IsPassiveModeEnabled"]==1]["RealTimeProtectionState"].value_counts().idxmax())

#5 How many systems have a screen resolution of 1366 x 768?
print(df1[(df1["PrimaryDisplayResolutionHorizontal"]==1366) & (df1["PrimaryDisplayResolutionVertical"]==768)].shape[0])

#6 What is the 50th percentile value of “TotalPhysicalRAMMB”?
print(df1["TotalPhysicalRAMMB"].median())

# **MILESTONE 2**

In [ ]:
import scipy.stats as stats
from sklearn.preprocessing import OneHotEncoder,MinMaxScaler,OrdinalEncoder
from sklearn.impute import SimpleImputer
from sklearn.model_selection import train_test_split
from sklearn.linear_model import SGDClassifier
from sklearn.metrics import accuracy_score

df1=df.copy()
X=df1.drop("target",axis=1)
y=df["target"]

#1 Identify the redundant columns in the training dataset based on the values taken.
redundent_cols=[col for col in df1.columns if df1[col].nunique()==1]
print(redundent_cols)

#2 Determine the columns among the given set which have the highest positive correlation

def cramers_v(x, y):
    confusion_matrix = pd.crosstab(x, y)
    chi2 = stats.chi2_contingency(confusion_matrix)[0]
    n = confusion_matrix.sum().sum()
    r, k = confusion_matrix.shape
    return np.sqrt(chi2 / (n * (min(r, k) - 1)))

print(cramers_v(df1['DateAS'], df1['SignatureVersion']))
print(cramers_v(df1['OSBuildLab'], df1['NumericOSVersion']))
print(cramers_v(df1['OSEdition'], df1['OSSkuFriendlyName']))
print(cramers_v(df1['OSProductSuite'], df1['OSSkuFriendlyName']))

#3 Create a new dataframe called cat_df which contains only the columns of datatype  'object'. In cat_df, for all the columns which take less than or equal to 10 unique values, use OneHotEncoder(sparse=False, drop='first', handle_unknown='ignore'). What is the new number of columns in the cat_df dataframe?

cat_df=df.select_dtypes(include=["object"])
req_cols=[cols for cols in cat_df.columns if cat_df[cols].nunique()<=10]

encoder = OneHotEncoder(sparse=False, drop='first', handle_unknown='ignore')
encoded_df = pd.DataFrame(encoder.fit_transform(cat_df[req_cols]), columns=encoder.get_feature_names_out(req_cols))

cat_df = cat_df.drop(columns=req_cols)
cat_df = pd.concat([cat_df, encoded_df], axis=1)
print(cat_df.shape[1])

#4 Create a new dataframe called num_df which contains only the columns of datatype  'int64' and 'float64'. Use MinMaxScaler() on num_df. What is the sum of all the values in num_df?

num_df = df.select_dtypes(include=['int64', 'float64'])

scaler = MinMaxScaler()
num_df_scaled = pd.DataFrame(scaler.fit_transform(num_df), columns=num_df.columns)

print(num_df_scaled.sum().sum()) #Required Sum

#5
#imputing all features
imputer = SimpleImputer(strategy='most_frequent')
X_imputed=pd.DataFrame(imputer.fit_transform(X),columns=X.columns)

categorical_cols = X_imputed.select_dtypes(include=['object']).columns

#ordinal encoder on categorical columns
encoder = OrdinalEncoder()  
X_imputed[categorical_cols] = encoder.fit_transform(X_imputed[categorical_cols])

#train-test-split
X_train, X_test, y_train, y_test = train_test_split(X_imputed, y, test_size=0.2, random_state=42)

#SGDClassifier
clf = SGDClassifier(random_state=42)
clf.fit(X_train, y_train)

#Accuracy
y_pred = clf.predict(X_test)
print(accuracy_score(y_test, y_pred))


# **MILESTONE 3**

In [ ]:
from sklearn.preprocessing import StandardScaler, LabelEncoder
from sklearn.impute import SimpleImputer
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.model_selection import train_test_split
from sklearn.metrics import accuracy_score

# Copy the dataset
df1 = df.copy()

# Define features and target
X = df1.drop(columns=['target'])  
y = df1['target']  

# Select numerical and categorical columns
num_cols = X.select_dtypes(include=['int64', 'float64']).columns.tolist()
cat_cols = X.select_dtypes(include=['object']).columns.tolist()

for col in cat_cols:
    le = LabelEncoder()
    X[col] = le.fit_transform(X[col])

cat_pipe=Pipeline(steps=[
    ('impute',SimpleImputer(strategy='most_frequent')),
])

num_pipe=Pipeline(steps=[
    ('impute',SimpleImputer(strategy='mean')),
    ('scaler',StandardScaler())
])

transform=ColumnTransformer(transformers=[
    ('cat',cat_pipe,cat_cols),
    ('num',num_pipe,num_cols)
],remainder='passthrough')

transformed_X=transform.fit_transform(X)

# Train-test split
X_train, X_test, y_train, y_test = train_test_split(transformed_X, y, test_size=0.2, random_state=42)

# Print Shapes
print("Train shape:", X_train.shape)
print("Test shape:", X_test.shape)

# Fit a Decision tree model (random state 42) on the training set and perform hyper parameter tuning using grid search with 3 folds and use scoring as accuracy

from sklearn.tree import DecisionTreeClassifier
from sklearn.model_selection import GridSearchCV

dt_model = DecisionTreeClassifier(random_state=42)

# Hyperparameters for tuning
param_grid = {
    'max_depth': [20, 30],
    'min_samples_split': [2, 5],
    'min_samples_leaf': [1, 2]
}

# Perform Grid Search with 3-fold cross-validation
grid_search = GridSearchCV(
    estimator=dt_model,
    param_grid=param_grid,
    cv=3, 
    scoring='accuracy',
    n_jobs=-1  
)

grid_search.fit(X_train, y_train)

best_params=grid_search.best_params_
#1
print("Best max_depth:", best_params['max_depth'])
#2
print("Best min sample split",best_params['min_samples_split'])
#3
print("Best min sample leaf",best_params['min_samples_leaf'])
#4
best_dt_model = grid_search.best_estimator_
y_pred = best_dt_model.predict(X_test)

print(accuracy_score(y_test, y_pred))


In [ ]:
#4 Fit an AdaBoostClassifier model (random state 42) on the training set and perform hyper parameter tuning using grid search with 3 folds and use scoring as accuracy
from sklearn.ensemble import AdaBoostClassifier
from sklearn.model_selection import GridSearchCV
from sklearn.metrics import accuracy_score

adaboost=AdaBoostClassifier(random_state=42)

params_grid={
    'n_estimators': [10, 20, 30],
    'learning_rate': [5, 10],
    'algorithm': ['SAMME']
}

grid_search=GridSearchCV(
    estimator=adaboost,
    param_grid=params_grid,
    cv=2,
    scoring="accuracy",
    n_jobs=-1)
grid_search.fit(X_train,y_train)

best_params=grid_search.best_params_

#5
print(" Best estimator ",best_params['n_estimators'])
#6
print(" Best learning rate ",best_params['learning_rate'])
#7
best_ada_model=grid_search.best_estimator_
y_pred=best_ada_model.predict(X_test)
print("Accuracy ",accuracy_score(y_test,y_pred))

# **MILESTONE 4**

In [ ]:
import pandas as pd
from sklearn.preprocessing import OrdinalEncoder, StandardScaler
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline

#preprocessing
df1=df.dropna()
X = df1.drop(columns=['target'])
y = df1['target']

cat_cols = X.select_dtypes(include=['object', 'category']).columns
num_cols = X.select_dtypes(include=['int64', 'float64']).columns

# preprocessing pipeline
#   - Use OrdinalEncoder for categorical columns.
#   - Use StandardScaler for all columns.
preprocessor = Pipeline(steps=[
    ('ordinal_encoder', ColumnTransformer(
        transformers=[
        ('cat', OrdinalEncoder(), cat_cols)
        ],
        remainder='passthrough'
    )),
    ('scaler', StandardScaler())
])
X_processed = preprocessor.fit_transform(X)

**PCA**

In [ ]:
from sklearn.decomposition import PCA
from sklearn.feature_selection import SelectKBest
from sklearn.metrics import mean_squared_error

pca=PCA()
X_pca=pca.fit_transform(X_processed)

explained_variance = np.cumsum(pca.explained_variance_ratio_)
#1
n_comp=np.argmax(explained_variance >= 0.70) + 1
print(n_comp)

#2
pca_2 = PCA(n_components=n_comp)
X_pca_2 = pca_2.fit_transform(X_processed)

# Reconstruct the data
X_reconstructed = pca_2.inverse_transform(X_pca_2)

mse = mean_squared_error(X_processed, X_reconstructed)
print(mse)

#3
pca_40 = PCA(n_components=40)
X_pca_40 = pca_40.fit_transform(X_processed)

variance_explained_40 = np.sum(pca_40.explained_variance_ratio_)
print(variance_explained_40)

In [ ]:
from sklearn.decomposition import PCA
from sklearn.feature_selection import SelectKBest,f_classif

#4
selector = SelectKBest(score_func=f_classif, k=15)
selector.fit(X_processed, y)

X_processed_df = pd.DataFrame(X_processed, columns=X.columns)

selected_mask = selector.get_support()
selected_features = X_processed_df.columns[selected_mask]
print("Selected features:", selected_features.tolist())

#5
best_feature_score = np.nanmax(selector.scores_)
print("Best feature score:", best_feature_score)

# **Preprocessing**

In [ ]:
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import OrdinalEncoder, StandardScaler

def detectColumnTypes(X):
    X = X.loc[:, X.nunique() > 1]
    cat_cols=X.select_dtypes(include=['object']).columns
    num_cols=X.select_dtypes(include=['float64','int64']).columns
    return  X,cat_cols, num_cols
    
#Preprocessing for numerical features
num_preprocessor = Pipeline(steps=[
    ('imputer', SimpleImputer(strategy='median')), 
    ('scaler', StandardScaler())
])

# Preprocessing for categorical features
cat_preprocessor = Pipeline(steps=[
    ('imputer', SimpleImputer(strategy='most_frequent')),
    ('encoder', OrdinalEncoder(handle_unknown='use_encoded_value', unknown_value=-1)) 
])

X_cleaned, cat_cols, num_cols = detectColumnTypes(X)

# Column transformer pipeline
preprocessor = ColumnTransformer(
    transformers=[
        ('num', num_preprocessor, num_cols),
        ('cat', cat_preprocessor, cat_cols)
    ]
)

# Apply preprocessing to the dataset
X_preprocessed = preprocessor.fit_transform(X_cleaned, y)
X_preprocessed.shape

# **Logistic Regression**

Hyperparamter Tuning

In [ ]:

from sklearn.model_selection import GridSearchCV
from sklearn.linear_model import LogisticRegression
from sklearn.model_selection import train_test_split

X_train, X_test, y_train, y_test = train_test_split(X_preprocessed, y, test_size=0.3, random_state=42)

# Parameter Grid
param_grid = {
    'penalty': ['l2'],  
    'C': [0.001, 0.01, 0.1, 1], 
    'solver': ['newton-cg']  
}

# Logistic Regression Model
logreg = LogisticRegression() 

# RandomizedSearchCV
grid_search = GridSearchCV(logreg, param_grid=param_grid,cv=3, scoring='accuracy', n_jobs=-1)
grid_search.fit(X_train, y_train)

print("Best parameters:", grid_search.best_params_)
print("Best score:", grid_search.best_score_)

In [ ]:
from sklearn.metrics import accuracy_score,classification_report,confusion_matrix
from sklearn.model_selection import train_test_split

X_train, X_test, y_train, y_test = train_test_split(X_preprocessed, y, test_size=0.3, random_state=42)
# Initialize the Logistic Regression model
lr_model = grid_search.best_estimator_

# Fit the model on the training data
lr_model.fit(X_train, y_train)

# Make predictions on the test set
y_pred = lr_model.predict(X_test)

# Evaluate the model
accuracy = accuracy_score(y_test, y_pred)
print(f'Accuracy: {accuracy * 100:.2f}%')
print(classification_report(y_test,y_pred))
print(confusion_matrix(y_test,y_pred))
# Load the test data from the CSV file
X_testf = pd.read_csv('/kaggle/input/System-Threat-Forecaster/test.csv')

# Apply the same preprocessing to the test data
X_testf = preprocessor.transform(X_testf)

# Make predictions on the external test data
y_testf = lr_model.predict(X_testf)

# Create the submission DataFrame
submission = pd.DataFrame({"id": range(0, X_testf.shape[0]), "target": y_testf})

# Save the submission to a CSV file
submission.to_csv('submission.csv', index=False)

# Print the shape of the submission
print(submission.shape)

# DecisionTreeClassifier

In [ ]:
from sklearn.tree import DecisionTreeClassifier
from sklearn.model_selection import RandomizedSearchCV, train_test_split
from sklearn.metrics import accuracy_score
import numpy as np

# Split the dataset into training and testing sets
X_train, X_test, y_train, y_test = train_test_split(X_preprocessed, y, test_size=0.3, random_state=16)

# Define hyperparameter space
param_dist = {
    'max_depth': np.arange(5, 30, 5),  
    'min_samples_split': np.arange(2, 50, 5), 
    'min_samples_leaf': np.arange(1, 20, 2),  
    'max_features': ['sqrt', 'log2', None],  
    'criterion': ['gini', 'entropy'] 
}

# Initialize Decision Tree model
dt_model = DecisionTreeClassifier(random_state=42)

# Use RandomizedSearchCV for hyperparameter tuning
random_search = RandomizedSearchCV(
    dt_model, 
    param_distributions=param_dist,
    n_iter=20, 
    cv=3,  
    scoring='accuracy',
    random_state=42,
    n_jobs=-1  
)

# Perform hyperparameter search
random_search.fit(X_train, y_train)

# Print best parameters
print("Best parameters:", random_search.best_params_)
print("Best accuracy:", random_search.best_score_)

# Train Decision Tree with best parameters
best_dt = random_search.best_estimator_
y_pred = best_dt.predict(X_test)

# Evaluate performance
accuracy = accuracy_score(y_test, y_pred)
print(f'Final Test Accuracy: {accuracy * 100:.2f}%')
print(classification_report(y_test,y_pred))
print(confusion_matrix(y_test,y_pred))

# Random Forest Classifier

In [ ]:
import pandas as pd
import numpy as np
from sklearn.ensemble import RandomForestClassifier
from sklearn.model_selection import train_test_split
from sklearn.feature_selection import SelectFromModel
from sklearn.metrics import accuracy_score

# Split the dataset
X_train, X_test, y_train, y_test = train_test_split(X_preprocessed, y, test_size=0.3, random_state=16)

#Random Forest model
rf_model = RandomForestClassifier(n_estimators=150, max_depth=20, random_state=42, n_jobs=-1)
rf_model.fit(X_train,y_train)

# Feature Selection using Random Forest
selector = SelectFromModel(rf_model, threshold="median",prefit=True)

# Apply Feature Selection to Training and Testing Sets
X_train_selected = selector.transform(X_train)
X_test_selected = selector.transform(X_test)

# Train the Model with Selected Features
rf_model.fit(X_train_selected, y_train)

# Predictions
y_pred = rf_model.predict(X_test_selected)

# Model Performance
accuracy = accuracy_score(y_test, y_pred)
print(f'✅ Random Forest Accuracy: {accuracy * 100:.2f}%')
print(classification_report(y_test,y_pred))
print(confusion_matrix(y_test,y_pred))


# **XGBoost**

In [ ]:
from xgboost import XGBClassifier
from sklearn.model_selection import train_test_split,RandomizedSearchCV
from sklearn.feature_selection import SelectFromModel
from sklearn.metrics import accuracy_score,classification_report,confusion_matrix

# Split the dataset
X_train, X_test, y_train, y_test = train_test_split(X_preprocessed, y, test_size=0.2, random_state=16)

param_dist = {
    'n_estimators': [50, 100, 150, 200, 250],
    'max_depth': [6, 8, 10],
    'learning_rate': [0.01, 0.05, 0.1]
}

xgb_model = XGBClassifier(random_state=42, n_jobs=-1)
random_search = RandomizedSearchCV(xgb_model, param_dist, cv=3, scoring='accuracy',n_jobs=-1)
random_search.fit(X_train, y_train)

best_params=random_search.best_params_
print(f'best parameters are {best_params}')

# Train XGBoost Model Before Feature Selection
xgb_model = XGBClassifier(**best_params, random_state=42, n_jobs=-1)
xgb_model.fit(X_train, y_train)
# Feature Selection using XGBoost
selector = SelectFromModel(xgb_model, threshold="median", prefit=True)

# Apply Feature Selection to Training and Testing Sets
X_train_selected = selector.transform(X_train)
X_test_selected = selector.transform(X_test)

# Retrain the XGBoost Model with Selected Features
xgb_selected_model = XGBClassifier(**best_params, random_state=42, n_jobs=-1)
xgb_selected_model.fit(X_train_selected, y_train)

# Predictions
y_pred = xgb_selected_model.predict(X_test_selected)

# Model Performance
accuracy = accuracy_score(y_test, y_pred)
print(f'✅ XGBoost Accuracy: {accuracy * 100:.2f}%')
print(classification_report(y_test,y_pred))
print(confusion_matrix(y_test,y_pred))

# Load External Test Data
X_testf = pd.read_csv('/kaggle/input/System-Threat-Forecaster/test.csv')

# Apply the same preprocessing to external test data
X_testf_transformed = preprocessor.transform(X_testf)

# **Fix: Apply Feature Selection to Test Data**
X_testf_selected = selector.transform(X_testf_transformed)

# Make Predictions
y_testf = xgb_selected_model.predict(X_testf_selected)

# Create Submission DataFrame
submission = pd.DataFrame({"id": range(0, X_testf.shape[0]), "target": y_testf})

# Save Submission
submission.to_csv('submission.csv', index=False)